### ***Phase 1: Environment Setup and Model Loading***

**What we do here:** Install essential libraries (Transformers, ChromaDB, Sentence-Transformers) and download the `google/flan-t5-base` model directly to the GPU.

In [1]:
!pip install transformers torch sentencepiece chromadb sentence-transformers

In [2]:
!pip install accelerate

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Downloading the model... this takes a minute.")
model_name = "google/flan-t5-base"

# Load tokenizer and model, pushing to GPU (cuda) for speed
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model is ready!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model is ready!


### ***Phase 2: Building the RAG Knowledge Base***

**What we do here:** Set up a local vector database using ChromaDB and `sentence-transformers` to store fitness facts, allowing the model to search and retrieve relevant context accurately.

In [4]:
import chromadb
from sentence_transformers import SentenceTransformer

# Our small knowledge base
fitness_facts = [
    "For fat loss, a caloric deficit of 300 to 500 calories below maintenance supports steady fat reduction while preserving lean mass.",
    "For muscle hypertrophy during a bulk, a slight surplus of 250 to 400 calories above maintenance optimizes growth while limiting fat gain.",
    "Daily protein intake for resistance training should range between 1.6g and 2.2g per kilogram of total body weight.",
    "Dietary fat should account for 20% to 30% of total daily calories to regulate hormone production, including testosterone.",
    "Carbohydrates are the primary fuel source for anaerobic glycolysis; consume 3g to 7g per kilogram depending on training volume.",
    "Body recomposition is achievable at maintenance calories with high protein (2.0g-2.4g/kg) and progressive overload training.",
    "Iron and Vitamin C should be consumed together to maximize non-heme iron absorption and maintain red blood cell oxygen transport.",
    "Zinc and Magnesium (ZMA) play crucial roles in protein synthesis, immune response, and central nervous system recovery during sleep.",
    "Creatine monohydrate supplementation at 3g to 5g daily saturates intramuscular phosphocreatine stores, enhancing peak power output.",
    "Hydration directly impacts protein synthesis; drink 35ml to 45ml of water per kilogram of body weight daily."
] 

print("Setting up the database...")
db_client = chromadb.Client()
collection = db_client.create_collection(name="gym_knowledge")

print("Loading the text-to-number model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Saving facts to the database...")

for i, fact in enumerate(fitness_facts):
    embedding_numbers = embed_model.encode(fact).tolist()
    collection.add(
        embeddings=[embedding_numbers],
        documents=[fact],
        ids=[str(i)]
    )
print(f"Loaded {len(fitness_facts)} curated facts into ChromaDB!")

Setting up the database...
Loading the text-to-number model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Saving facts to the database...
Loaded 10 curated facts into ChromaDB!


In [5]:
user_question = "How much protein should I eat if I weigh 80 kg?"
print(f"User asked: {user_question}")

# Search the database for the most relevant fact
question_numbers = embed_model.encode(user_question).tolist()
search_results = collection.query(
    query_embeddings=[question_numbers],
    n_results=1
)

found_fact = search_results['documents'][0][0]
print(f"\nFound this fact in database: {found_fact}")

# Give the fact and the question to the AI model
prompt = f"Use this fact to answer the question.\nFact: {found_fact}\nQuestion: {user_question}\nAnswer:"

# Generate the final human-readable answer
inputs = tokenizer(prompt, return_tensors="pt")

# Move input tensors to the same device as the model
device = next(model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model.generate(**inputs, max_length=50)
final_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"\nFitGen AI Answer: {final_answer}")

User asked: How much protein should I eat if I weigh 80 kg?

Found this fact in database: Daily protein intake for resistance training should range between 1.6g and 2.2g per kilogram of total body weight.

FitGen AI Answer: 2.2g


### ***Phase 3: Baseline Testing and Evaluation***

**What we do here:** Test the default model's output and evaluate it using Hugging Face's `evaluate` library (ROUGE and BLEU scores) to establish a baseline before fine-tuning.


In [6]:
!pip install evaluate rouge_score

In [7]:
import numpy as np
import evaluate

# 1. Define predictions and references first
ai_prediction = final_answer
ideal_reference = "If you weigh 80 kg, you should aim for about 160 grams of protein daily."

print(f"AI Answer: {ai_prediction}")
print(f"Ideal Answer: {ideal_reference}\n")

# 2. Semantic Cosine Similarity
def compute_semantic_similarity(pred, ref):
    emb_pred = embed_model.encode(pred)
    emb_ref = embed_model.encode(ref)
    return float(np.dot(emb_pred, emb_ref) / (np.linalg.norm(emb_pred) * np.linalg.norm(emb_ref)))

sem_score = compute_semantic_similarity(ai_prediction, ideal_reference)
print(f"Semantic Cosine Similarity: {sem_score:.4f}")

# 3. Load Graders and Compute ROUGE / BLEU
print("\nLoading AI Graders...")
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

rouge_result = rouge.compute(predictions=[ai_prediction], references=[ideal_reference])
print(f"ROUGE-1 Score (Keyword Match): {rouge_result['rouge1']:.4f}")

bleu_result = bleu.compute(predictions=[ai_prediction], references=[[ideal_reference]])
print(f"BLEU Score (Exact Phrasing Match): {bleu_result['bleu']:.4f}")

AI Answer: 2.2g
Ideal Answer: If you weigh 80 kg, you should aim for about 160 grams of protein daily.

Semantic Cosine Similarity: 0.1655

Loading AI Graders...
ROUGE-1 Score (Keyword Match): 0.0000
BLEU Score (Exact Phrasing Match): 0.0000


### ***Phase 4: Parameter-Efficient Fine-Tuning (PEFT / LoRA)***

**What we do here:** Attach a LoRA adapter to train less than 0.4% of the model's parameters, teaching it to output complete, conversational sentences using structured RAG.

In [8]:
!pip install peft

In [9]:
!pip install --upgrade torchao

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 2.3 MB/s eta 0:00:01
   ------------------------------ --------- 1.0/1.4 MB 2.7 MB/s eta 0:00:01
   ------------------------------ --------- 1.0/1.4 MB 2.7 MB/s eta 0:00:01
   -------------------------------------- - 1.3/1.4 MB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 1.3 MB/s  0:00:01


In [10]:
from peft import LoraConfig, get_peft_model, TaskType

# 1. Create the LoRA rules
# We tell it we are doing text-to-text generation (SEQ_2_SEQ_LM)
lora_rules = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,                         # Increased capacity for style and nutrition coaching
    lora_alpha=32,                # Proportional scaling (2 * r)
    target_modules=["q", "v"],    # Explicit attention projections for Flan-T5
    lora_dropout=0.05,
    bias="none"
)
# 2. Attach the rules to our existing model
peft_model = get_peft_model(model, lora_rules)

# 3. See how much memory we saved!
peft_model.print_trainable_parameters()

W0929 16:45:35.115000 20180 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


trainable params: 1,769,472 || all params: 249,347,328 || trainable%: 0.7096


In [11]:
from peft import LoraConfig, get_peft_model, TaskType

print("Setting up LoRA (The sticky note)...")


lora_rules = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,               # The size of our sticky note (keep it small)
    lora_alpha=16,     # How strongly to apply our new rules
    lora_dropout=0.1,  # Helps prevent the AI from memorizing too much
    bias="none"
)

# 2. Attach the rules to our existing model
peft_model = get_peft_model(model, lora_rules)

# 3. See how much memory we saved!
peft_model.print_trainable_parameters()

Setting up LoRA (The sticky note)...


C:\Users\chiku\anaconda3\Lib\site-packages\peft\mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
C:\Users\chiku\anaconda3\Lib\site-packages\peft\tuners\tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


In [12]:
!pip install datasets

# ***Train the model (Fine-tuning)***

### ***Phase 5: Final Testing and Saving Model Weights***

**What we do here:** Test the fine-tuned FitGen AI to verify it now responds in full, natural language sentences, and save the custom LoRA weights to disk for future deployment.

In [13]:
%pip install "accelerate>=1.1.0"

Note: you may need to restart the kernel to use updated packages.


In [14]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import Dataset

# 1. Expanded RAG training dataset covering key fitness & nutrition intents
training_data = {
    "prompt": [
        "Use this fact to answer the question.\nFact: A person weighing 80 kg should aim for about 160 grams of protein daily to build muscle.\nQuestion: How much protein should I eat if I weigh 80 kg?\nAnswer:",
        "Use this fact to answer the question.\nFact: For fat loss, a caloric deficit of 300 to 500 calories below maintenance supports steady fat reduction while preserving lean mass.\nQuestion: How many calories should I cut to lose fat?\nAnswer:",
        "Use this fact to answer the question.\nFact: For muscle hypertrophy during a bulk, a slight surplus of 250 to 400 calories above maintenance optimizes growth while limiting fat gain.\nQuestion: How much should I eat to bulk up cleanly?\nAnswer:",
        "Use this fact to answer the question.\nFact: Dietary fat should account for 20% to 30% of total daily calories to regulate hormone production, including testosterone.\nQuestion: Why do I need fats if I want to get lean?\nAnswer:",
        "Use this fact to answer the question.\nFact: Zinc and Magnesium (ZMA) play crucial roles in protein synthesis, immune response, and central nervous system recovery during sleep.\nQuestion: What micronutrients help with workout recovery?\nAnswer:",
        "Use this fact to answer the question.\nFact: A good pre-workout snack is a banana or a slice of toast with peanut butter.\nQuestion: What is a good pre-workout snack?\nAnswer:",
        "Use this fact to answer the question.\nFact: Rest days are important because muscles actually grow when you rest, not when you train.\nQuestion: Why are rest days important?\nAnswer:",
        "Use this fact to answer the question.\nFact: Creatine monohydrate supplementation at 3g to 5g daily saturates intramuscular phosphocreatine stores, enhancing peak power output.\nQuestion: What does creatine do for strength?\nAnswer:"
    ],
    "answer": [
        "If you weigh 80 kg, you should aim for about 160 grams of protein daily to build and repair muscle.",
        "Aim for a moderate caloric deficit of 300 to 500 calories below maintenance to steadily lose fat while protecting lean muscle.",
        "Eat in a slight surplus of 250 to 400 calories above maintenance to optimize muscle growth while minimizing unwanted fat gain.",
        "Ensure 20% to 30% of your daily intake comes from healthy fats to support vital hormone production and joint recovery.",
        "Prioritize zinc and magnesium in your diet to support muscle protein synthesis and promote nervous system recovery during sleep.",
        "A great pre-workout snack is a banana or a slice of toast with peanut butter for quick, sustained energy.",
        "Rest days are essential because muscle tissue recovers and grows during downtime, not while lifting in the gym.",
        "Take 3 to 5 grams of creatine monohydrate daily to saturate phosphocreatine stores and boost maximum lifting power."
    ]
}

my_dataset = Dataset.from_dict(training_data)

# 2. Tokenize inputs and mask padding in labels
def prepare_data(batch):
    inputs = tokenizer(batch["prompt"], padding="max_length", max_length=128, truncation=True)
    targets = tokenizer(batch["answer"], padding="max_length", max_length=64, truncation=True)

    # Replace padding token ids with -100 so the model loss ignores padding
    labels = [
        [(token if token != tokenizer.pad_token_id else -100) for token in target]
        for target in targets["input_ids"]
    ]
    inputs["labels"] = labels
    return inputs

ready_dataset = my_dataset.map(prepare_data, batched=True)

# 3. Configure training parameters
print("Setting up the training gym...")
training_args = Seq2SeqTrainingArguments(
    output_dir="./fitgen_coach",
    learning_rate=5e-4,
    per_device_train_batch_size=2,
    num_train_epochs=8,
    weight_decay=0.01,
    logging_steps=2,
    save_strategy="no"
)

trainer = Seq2SeqTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=ready_dataset,
)

# 4. Run fine-tuning and automatically persist weights
print("Starting the workout (training)...")
trainer.train()
print("Training complete!")

peft_model.save_pretrained("./fitgen_final_weights")
print("Weights successfully saved to ./fitgen_final_weights")

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Setting up the training gym...
Starting the workout (training)...


C:\Users\chiku\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
2,2.827279
4,2.343028
6,2.301041
8,2.667346
10,2.934701
12,1.884870
14,2.590665
16,1.963122
18,1.829332
20,2.637746


Training complete!
Weights successfully saved to ./fitgen_final_weights


In [16]:
# The same question and fact from our database earlier
user_question = "How much protein should I eat if I weigh 80 kg?"
found_fact = "A person weighing 80 kg should aim for about 160 grams of protein daily to build muscle."

# The prompt
prompt = f"Use this fact to answer the question.\nFact: {found_fact}\nQuestion: {user_question}\nAnswer:"

# Put the inputs on the GPU
inputs = tokenizer(prompt, return_tensors="pt").to(peft_model.device)

# Generate the answer using our fine-tuned peft_model
outputs = peft_model.generate(**inputs, max_length=50)
new_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- BEFORE TRAINING ---")
print("AI Answer: 160 grams\n")

print("--- AFTER LoRA TRAINING ---")
print(f"FitGen AI Answer: {new_answer}")

--- BEFORE TRAINING ---
AI Answer: 160 grams

--- AFTER LoRA TRAINING ---
FitGen AI Answer: 160 grams daily.


In [17]:
# Save the model weights
peft_model.save_pretrained("./fitgen_final_weights")
print("Model weights successfully saved to disk!")

Model weights successfully saved to disk!
